# Lab 2 — Task 1: Music Genre Classification with CNNs

Este notebook desarrolla exclusivamente la **Task 1**: entrenar una CNN desde cero para clasificar géneros musicales a partir de imágenes de espectrogramas.

Se ejecutarán **17 configuraciones**: BASE0, seis experimentos de profundidad y diez experimentos de factores. Cada configuración se entrenará durante **50 épocas completas**, sin parada anticipada. Se conservarán los pesos de la época con mejor *validation accuracy*.

**Metodología de evaluación:**
1. Se cargan las imágenes organizadas por carpetas, una carpeta por género.
2. Se realiza una partición estratificada fija: 64 % entrenamiento, 16 % validación y 20 % test.
3. Todos los experimentos utilizan la misma partición para que la comparación sea justa.
4. La selección del modelo se basa exclusivamente en *validation accuracy*.
5. El conjunto de test se evalúa una sola vez, después de seleccionar el mejor modelo.

Los experimentos de profundidad cambian la secuencia de filtros. Los experimentos de factores modifican un solo parámetro respecto a BASE0. La resolución 192×288 se ejecuta al final porque tiene mayor coste computacional.

La **Task 2 (VGG16)** se mantiene en el notebook independiente `Lab2_Task2_VGG16_Music_Genre.ipynb`.

## 1. Imports y configuración

En esta sección se importan las librerías para leer y procesar imágenes, construir la CNN con TensorFlow/Keras, dividir los datos, calcular métricas y visualizar resultados. Se fijan las semillas aleatorias para hacer la ejecución más reproducible.

In [ ]:
# Si alguna librería no está instalada, descomenta y ejecuta esta línea:
# %pip install tensorflow scikit-learn matplotlib pandas

# Librerías estándar para rutas, aleatoriedad y liberar memoria.
import os
import random
import gc

# Procesamiento numérico y visualización.
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Framework para cargar datos y construir/entrenar la CNN.
import tensorflow as tf

# Utilidades para dividir el dataset y evaluar el clasificador.
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from IPython.display import display

# Fijar semillas para favorecer la reproducibilidad.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)

## 2. Cargar las imágenes y separar los datos

Cada subcarpeta de `images_original` representa un género musical. Primero se localizan las imágenes y se asigna una etiqueta numérica a cada género. Después se separa un 20 % para test y, del 80 % restante, un 20 % para validación.

La partición resultante es aproximadamente 64 % entrenamiento, 16 % validación y 20 % test. Se utiliza `stratify` para mantener la proporción de géneros en los tres subconjuntos. El conjunto de test queda reservado hasta la evaluación final.

In [ ]:
# Ruta al directorio con una subcarpeta por género musical.
# En Colab puedes usar, por ejemplo: '/content/drive/MyDrive/GTZAN/images_original'
DATA_DIR = 'Data/images_original'  # Cambia esta ruta si tus datos están en otro lugar.
# Tamaño del lote usado para entrenar y evaluar.
BATCH_SIZE = 32
# Todas las configuraciones se entrenan durante 50 épocas completas.
MAX_EPOCHS = 50

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"No encuentro {DATA_DIR}. Cambia DATA_DIR para apuntar a images_original."
    )

# Detectar las clases a partir de los nombres de las subcarpetas.
class_names = sorted(
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
)
extensions = ('.png', '.jpg', '.jpeg', '.bmp')
paths, labels = [], []

# Recorrer cada género y registrar la ruta y la etiqueta de cada imagen.
for label, genre in enumerate(class_names):
    folder = os.path.join(DATA_DIR, genre)
    for filename in sorted(os.listdir(folder)):
        if filename.lower().endswith(extensions):
            paths.append(os.path.join(folder, filename))
            labels.append(label)

paths = np.asarray(paths)
labels = np.asarray(labels)

if len(paths) == 0:
    raise ValueError("No se han encontrado imágenes. Comprueba la estructura de carpetas.")

print("Géneros:", class_names)
print("Total de imágenes:", len(paths))

# Separar el test antes de comenzar los experimentos: no se usará para elegir modelos.
dev_paths, test_paths, dev_labels, test_labels = train_test_split(
    paths, labels, test_size=0.20, random_state=SEED, stratify=labels
)
# Extraer validación únicamente del conjunto de desarrollo restante.
train_paths, val_paths, train_labels, val_labels = train_test_split(
    dev_paths, dev_labels, test_size=0.20, random_state=SEED, stratify=dev_labels
)

print(f"Train: {len(train_paths)} | Validation: {len(val_paths)} | Test: {len(test_paths)}")
print("No se usa test para comparar configuraciones.")

## 3. Preparar datasets para distintas resoluciones

La función `load_image` lee cada archivo, decodifica la imagen en RGB y la redimensiona a la resolución del experimento. La función `make_dataset` crea un `tf.data.Dataset`, baraja solo los datos de entrenamiento, agrupa las imágenes en lotes y prepara el siguiente lote en paralelo.

Como algunos experimentos cambian la resolución, los datasets se construyen para cada configuración. Al final de esta sección se muestran unas imágenes de ejemplo para comprobar que la lectura y las etiquetas son correctas.

In [ ]:
# Leer una imagen, convertirla a RGB y redimensionarla al tamaño requerido.
def load_image(path, label, size):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, size)
    return image, label


# Crear un dataset por lotes. Solo se baraja el subconjunto de entrenamiento.
def make_dataset(file_paths, file_labels, size, training=False):
    ds = tf.data.Dataset.from_tensor_slices((file_paths, file_labels))

    if training:
        ds = ds.shuffle(
            buffer_size=len(file_paths),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    # Leer y transformar imágenes en paralelo para acelerar la entrada de datos.
    ds = ds.map(
        lambda path, label: load_image(path, label, size),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # Agrupar ejemplos y preparar lotes de forma anticipada.
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


# Visualizar cuatro imágenes de entrenamiento para verificar carga y etiquetas.
preview_ds = make_dataset(
    train_paths, train_labels, size=(128, 192), training=True
)

for images, labels_batch in preview_ds.take(1):
    plt.figure(figsize=(9, 3))
    for i in range(min(4, len(images))):
        plt.subplot(1, 4, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8))
        plt.title(class_names[int(labels_batch[i])])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# Liberar el dataset temporal de visualización.
del preview_ds

## 4. Definir BASE0 y los experimentos

BASE0 define la arquitectura y los hiperparámetros de referencia. Cada experimento de profundidad sustituye únicamente la secuencia de filtros; cada experimento de factores cambia una sola propiedad respecto a BASE0.

En los kernels, el primer número corresponde al eje vertical del espectrograma (frecuencia) y el segundo al eje horizontal (tiempo). Por eso `(7, 3)` es más alto en frecuencia y `(3, 7)` más ancho en tiempo.

Las 17 configuraciones se almacenan en un diccionario. La resolución 192×288 se mueve al final de la lista para ejecutar primero los experimentos menos costosos.

In [ ]:
# Configuración base. Se mantienen exactamente los parámetros indicados para BASE0.
BASE0 = dict(
    filters=(32, 64, 128),
    kernel=(3, 3),
    dropout=0.3,
    augment=True,
    lr=1e-3,
    size=(128, 192)
)

# Experimentos de profundidad: solo cambia la secuencia de filtros convolucionales.
DEPTH_EXPERIMENTS = {
    '2conv_16-32':             dict(filters=(16, 32)),
    '3conv_16-32-64':          dict(filters=(16, 32, 64)),
    '3conv_32-64-128':         dict(filters=(32, 64, 128)),
    '4conv_16-32-64-128':      dict(filters=(16, 32, 64, 128)),
    '4conv_32-64-128-256':     dict(filters=(32, 64, 128, 256)),
    '5conv_32-64-128-256-256': dict(filters=(32, 64, 128, 256, 256)),
}

# Experimentos de factores: cada uno modifica un único parámetro de BASE0.
FACTOR_EXPERIMENTS = {
    'dropout_0.0':     dict(dropout=0.0),
    'dropout_0.5':     dict(dropout=0.5),
    'no_augment':      dict(augment=False),
    'lr_3e-4':         dict(lr=3e-4),
    'lr_3e-3':         dict(lr=3e-3),
    'kernel_5x5':      dict(kernel=(5, 5)),
    'kernel_7x3_freq': dict(kernel=(7, 3)),  # Kernel más alto en frecuencia.
    'kernel_3x7_time': dict(kernel=(3, 7)),  # Kernel más ancho en tiempo.
    'size_64x96':      dict(size=(64, 96)),
    'size_192x288':    dict(size=(192, 288)),  # Más costoso: se ejecuta al final.
}


# Combinar BASE0 con los cambios de cada experimento sin modificar BASE0.
def merge_config(base, changes):
    config = base.copy()
    config.update(changes)
    return config


# Registrar primero la configuración base.
configurations = {'BASE0': BASE0.copy()}

# Añadir los experimentos de profundidad en el orden especificado.
for name, changes in DEPTH_EXPERIMENTS.items():
    configurations[name] = merge_config(BASE0, changes)

# Añadir los experimentos de factores en el orden especificado.
for name, changes in FACTOR_EXPERIMENTS.items():
    configurations[name] = merge_config(BASE0, changes)

# Mover la resolución más alta al final para reducir el coste inicial.
large_resolution = configurations.pop('size_192x288')
configurations['size_192x288'] = large_resolution

# Mostrar la tabla de configuraciones antes de entrenar.
config_table = pd.DataFrame.from_dict(
    configurations, orient='index'
).reset_index().rename(columns={'index': 'Configuration'})
config_table.index = range(1, len(config_table) + 1)

print(f'Número de configuraciones: {len(configurations)}')
display(config_table)

## 5. Construir, entrenar y seleccionar la CNN

La función `generate_model` construye una CNN desde cero usando los filtros, el kernel, el dropout, la resolución y la tasa de aprendizaje de cada configuración. La normalización de píxeles transforma los valores RGB al intervalo [0, 1]. El *data augmentation* se activa solo en las configuraciones que lo solicitan y solo afecta al entrenamiento.

Cada modelo se entrena durante **50 épocas completas**, sin *EarlyStopping*. Un callback guarda los pesos correspondientes a la época con mayor *validation accuracy*. Al terminar cada experimento, se compara esa métrica con la mejor obtenida hasta el momento. El test no se consulta en esta fase.

In [ ]:
# Construir la CNN a partir de los parámetros de la configuración recibida.
def generate_model(num_classes, config):
    height, width = config['size']
    inputs = tf.keras.Input(shape=(height, width, 3))
    x = inputs

    # Aplicar aumentación de datos solo cuando esté activada en la configuración.
    if config['augment']:
        x = tf.keras.layers.RandomTranslation(0.05, 0.05)(x)
        x = tf.keras.layers.RandomZoom(0.1)(x)
        x = tf.keras.layers.RandomContrast(0.1)(x)

    # Escalar los píxeles de [0, 255] a [0, 1].
    x = tf.keras.layers.Rescaling(1.0 / 255.0)(x)

    # Crear un bloque convolucional por cada número de filtros.
    for n_filters in config['filters']:
        x = tf.keras.layers.Conv2D(
            filters=n_filters,
            kernel_size=config['kernel'],
            padding='same',
            activation='relu'
        )(x)

        # Reducir a la mitad las dimensiones espaciales después de cada bloque.
        x = tf.keras.layers.MaxPooling2D(pool_size=2)(x)

    # Resumir cada mapa de características sin aplanar todos sus píxeles.
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    # Capa densa para combinar las características aprendidas.
    x = tf.keras.layers.Dense(128, activation='relu')(x)

    # Dropout opcional para reducir el sobreajuste.
    if config['dropout'] > 0:
        x = tf.keras.layers.Dropout(config['dropout'])(x)

    # Una probabilidad de salida por cada género musical.
    outputs = tf.keras.layers.Dense(
        num_classes, activation='softmax'
    )(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    # Adam optimiza los pesos y la entropía cruzada mide el error de clasificación.
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=config['lr']
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


# Callback personalizado: registra y guarda los pesos de la mejor época de validación.
# No detiene el entrenamiento; todas las configuraciones completan 50 épocas.
class SaveBestWeights(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.best_val_accuracy = -1.0
        self.best_weights = None
        self.best_epoch = 0

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_accuracy = logs.get('val_accuracy')

        if val_accuracy is not None and val_accuracy > self.best_val_accuracy:
            self.best_val_accuracy = float(val_accuracy)
            self.best_weights = [
                weight.copy() for weight in self.model.get_weights()
            ]
            self.best_epoch = epoch + 1


# Inicializar el registro de resultados y la mejor configuración global.
num_classes = len(class_names)
experiment_results = {}

best_name = None
best_val_accuracy = -1.0
best_weights = None
best_config = None

# Entrenar cada configuración con la misma partición de entrenamiento y validación.
for index, (name, config) in enumerate(
    configurations.items(), start=1
):
    print(f"\n{'=' * 75}")
    print(f"[{index}/{len(configurations)}] Entrenando: {name}")
    print(f"Parámetros: {config}")
    print(f"Épocas máximas: {MAX_EPOCHS}")
    print(f"{'=' * 75}")

    # Limpiar recursos entre modelos para limitar el consumo de memoria.
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(SEED)

    # Crear los datasets a la resolución de esta configuración.
    train_ds = make_dataset(
        train_paths, train_labels, config['size'], training=True
    )
    val_ds = make_dataset(
        val_paths, val_labels, config['size']
    )

    # Construir y entrenar el modelo durante las 50 épocas completas.
    candidate = generate_model(num_classes, config)
    save_best = SaveBestWeights()

    history = candidate.fit(
        train_ds,
        validation_data=val_ds,
        epochs=MAX_EPOCHS,
        shuffle=False,  # El barajado ya se realiza en train_ds.
        callbacks=[save_best],
        verbose=1
    )

    # Restaurar los pesos de la época que obtuvo la mejor validación.
    candidate.set_weights(save_best.best_weights)

    # Guardar métricas, curvas, configuración y número de épocas.
    result = {
        'config': config.copy(),
        'history': {
            key: list(values)
            for key, values in history.history.items()
        },
        'val_accuracy': save_best.best_val_accuracy,
        'best_epoch': save_best.best_epoch,
        'epochs': len(history.history['loss']),
        'train_accuracy': float(
            history.history['accuracy'][save_best.best_epoch - 1]
        )
    }
    experiment_results[name] = result

    print(
        f"Mejor validation accuracy: {result['val_accuracy']:.4f} "
        f"(época {result['best_epoch']})"
    )
    print(f"Épocas entrenadas: {result['epochs']}")

    # Actualizar el mejor modelo global usando exclusivamente validación.
    if result['val_accuracy'] > best_val_accuracy:
        best_val_accuracy = result['val_accuracy']
        best_name = name
        best_config = config.copy()
        best_weights = [
            weight.copy() for weight in candidate.get_weights()
        ]

    # Eliminar los objetos del experimento actual antes del siguiente.
    del candidate, history, train_ds, val_ds, save_best
    tf.keras.backend.clear_session()
    gc.collect()

# Resumen de la configuración seleccionada antes de evaluar test.
print("\n" + "=" * 75)
print("TODOS LOS EXPERIMENTOS HAN TERMINADO")
print("Configuración seleccionada:", best_name)
print(f"Mejor validation accuracy: {best_val_accuracy:.4f}")
print("Parámetros seleccionados:", best_config)

## 6. Comparar resultados y evaluar el modelo final

Primero se ordenan las 17 configuraciones por su mejor *validation accuracy*, mostrando también la época en la que se obtuvo esa métrica y las épocas entrenadas. Después se reconstruye el modelo ganador y se restauran sus mejores pesos.

**Solo ahora se utiliza el conjunto de test**, una vez, para calcular la accuracy final, el informe por género y la matriz de confusión. Las gráficas posteriores resumen los resultados de validación y las curvas de entrenamiento del modelo seleccionado.

In [ ]:
# Crear una tabla con los resultados de todos los experimentos y ordenarla por validación.
results_rows = []

for name, run in experiment_results.items():
    results_rows.append({
        'Configuration': name,
        'Train accuracy at best epoch': run['train_accuracy'],
        'Best validation accuracy': run['val_accuracy'],
        'Best epoch': run['best_epoch'],
        'Epochs trained': run['epochs'],
        'Parameters': run['config']
    })

results_df = pd.DataFrame(results_rows).sort_values(
    'Best validation accuracy', ascending=False
).reset_index(drop=True)

print("Comparación de todas las configuraciones:")
display(results_df.round(4))

# Reconstruir el mejor modelo y restaurar los pesos guardados durante el entrenamiento.
tf.keras.backend.clear_session()
model = generate_model(num_classes, best_config)
model.set_weights(best_weights)

# Crear el dataset de test con la resolución de la configuración seleccionada.
test_ds = make_dataset(test_paths, test_labels, best_config['size'])

# Evaluar el conjunto de test una sola vez, después de seleccionar el modelo.
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print(f"\nModelo seleccionado: {best_name}")
print(f"Validation accuracy: {best_val_accuracy:.4f}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

# Obtener predicciones de test para las métricas detalladas.
y_true = test_labels
y_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

# Informe de precisión, recall y F1-score para cada género.
print("\nInforme de clasificación en test:")
print(classification_report(
    y_true,
    y_pred,
    labels=range(num_classes),
    target_names=class_names,
    zero_division=0
))

# Matriz de confusión: filas = género real, columnas = género predicho.
cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(
    cm, display_labels=class_names
).plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
ax.set_title(f'Matriz de confusión en test — {best_name}')
plt.tight_layout()
plt.show()

# Comparar visualmente la mejor validation accuracy de cada configuración.
sorted_results = sorted(
    experiment_results.items(),
    key=lambda item: item[1]['val_accuracy']
)
names = [name for name, _ in sorted_results]
scores = [run['val_accuracy'] for _, run in sorted_results]

plt.figure(figsize=(12, max(5, len(names) * 0.35)))
plt.barh(names, scores)
plt.xlabel('Best validation accuracy')
plt.title('Comparación de configuraciones (validación)')
plt.xlim(0, 1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Mostrar las curvas de accuracy y loss del modelo elegido.
selected_history = experiment_results[best_name]['history']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(selected_history['accuracy'], label='Train accuracy')
axes[0].plot(selected_history['val_accuracy'], label='Validation accuracy')
axes[0].set(
    title=f'{best_name} — Accuracy',
    xlabel='Época',
    ylabel='Accuracy'
)

axes[1].plot(selected_history['loss'], label='Train loss')
axes[1].plot(selected_history['val_loss'], label='Validation loss')
axes[1].set(
    title=f'{best_name} — Loss',
    xlabel='Época',
    ylabel='Loss'
)

for ax in axes:
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()